# Audit NJ population and ZIP/ZCTA identifiers
This audit does not change population, case data, weights or model results. Update raw/repo paths if necessary. All outputs are saved to Weikun/population_geography_audit.

In [ ]:
from pathlib import Path
import json
import pandas as pd

raw=Path(r'C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data')
repo=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
out=repo/'population_geography_audit'
out.mkdir(parents=True,exist_ok=True)
attrs=pd.read_csv(repo/'regional_attributes_outputs/njny_zcta_regional_attributes_2021.csv',dtype={'zcta':str})
nj=attrs.loc[attrs.state.eq('NJ')].set_index('zcta')
codes=sorted(nj.index[nj.population.isna()|nj.population.le(0)])
pop=pd.read_csv(raw/'NJNY_Population 2021.csv.csv',dtype=str)
pop['zcta']=pop.GEO_ID.str.extract(r'([0-9]{5})$')
assert not pop.zcta.dropna().duplicated().any()
pop=pop.loc[pop.zcta.isin(codes)].set_index('zcta')
gaz=pd.read_csv(raw/'2021_Gaz_zcta_national.txt',sep='\t',dtype=str)
gaz.columns=gaz.columns.str.strip()
gaz=gaz.set_index('GEOID')
x=pd.read_csv(raw/'nj_xwalk.csv.gz',dtype=str)
assert not x.tabblk2020.duplicated().any()
selected=x.loc[x.zcta.isin(codes)]
assert selected.stusps.eq('NJ').all()
case=pd.read_csv(repo/'real_nj_cleaning_outputs'/'Real NJ COVID Zip.csv',usecols=['zip','date','onset_record_count'],dtype={'zip':str},parse_dates=['date'])
r=pd.read_csv(repo/'njny_commute_outputs/njny_commute_zcta_weights_2021.csv',dtype={'origin_zcta':str,'destination_zcta':str})
r=r.loc[r.origin_state.eq('NJ')&r.destination_state.eq('NJ')].copy()
r['inbound_weight_nj']=r.commute_jobs/r.groupby('destination_zcta').commute_jobs.transform('sum')
assert r.origin_zcta.ne(r.destination_zcta).all()
counts=case.groupby('zip').onset_record_count.sum()
positive=case.loc[case.onset_record_count.gt(0)].groupby('zip').size()
rows=[]
for zz in codes:
    xx=selected.loc[selected.zcta.eq(zz)]
    rr=r.loc[r.origin_zcta.eq(zz)]
    row={'identifier':zz,'raw_population_geo_id':pop.loc[zz,'GEO_ID'],
         'raw_population_name':pop.loc[zz,'NAME'],
         'raw_population_estimate':int(pop.loc[zz,'B01003_001E']),
         'raw_population_margin_of_error':int(pop.loc[zz,'B01003_001M']),
         'merged_population':float(nj.loc[zz,'population']),
         'population_record_present':True,'population_value_status':'source_estimate_zero',
         'gazetteer_zcta_present':zz in gaz.index,
         'land_area_sq_km':float(gaz.loc[zz,'ALAND'])/1e6,
         'lodes_zcta_present':len(xx)>0,'lodes_mapped_blocks':len(xx),
         'lodes_county_names':'; '.join(sorted(xx.ctyname.dropna().unique())),
         'lodes_place_names':'; '.join(sorted(xx.stplcname.dropna().unique())),
         'lodes_crosswalk_createdates':'; '.join(sorted(xx.createdate.dropna().unique())),
         'included_onset_records_2020_2023':int(counts.get(zz,0)),
         'days_with_positive_recorded_onset':int(positive.get(zz,0)),
         'nj_external_outbound_jobs':int(rr.commute_jobs.sum()),
         'directly_connected_nj_destinations':int(rr.destination_zcta.nunique()),
         'population_join_check':'raw estimate equals merged population',
         'postal_zip_to_zcta_validation':'not established: supplied crosswalk maps blocks to ZCTAs, not postal ZIPs to ZCTAs'}
    assert row['raw_population_estimate']==row['merged_population']==0
    assert row['gazetteer_zcta_present'] and row['lodes_zcta_present']
    rows.append(row)
result=pd.DataFrame(rows)
result.to_csv(out/'source_population_geography_audit.csv',index=False,encoding='utf-8-sig')
bad=r.loc[r.origin_zcta.isin(codes)]
impact=bad.groupby('destination_zcta').agg(invalid_population_origin_jobs=('commute_jobs','sum'),invalid_population_weight=('inbound_weight_nj','sum'),invalid_origin_count=('origin_zcta','nunique')).reset_index()
impact['valid_population_source_weight']=1-impact.invalid_population_weight
impact=impact.sort_values('invalid_population_weight',ascending=False)
impact.to_csv(out/'affected_destination_weights.csv',index=False,encoding='utf-8-sig')
summary={'invalid_population_regions':len(codes),'missing_population_records':0,'zero_estimates_in_original_population_file':8,
         'all_eight_present_in_gazetteer_and_lodes':True,'population_join_value_mismatches':0,
         'postal_zip_to_zcta_mapping_validated':False,
         'invalid_population_origins_with_positive_external_nj_outflow':int(result.nj_external_outbound_jobs.gt(0).sum()),
         'affected_destinations':len(impact),'maximum_invalid_source_weight':float(impact.invalid_population_weight.max()),
         'median_invalid_source_weight':float(impact.invalid_population_weight.median()),
         'supplied_population_geography':'ZCTA (860Z200US identifiers), not a postal ZIP correspondence table',
         'supplied_lodes_crosswalk_geography':'2020 Census block to ZCTA; createdate in supplied file is 20251202',
         'no_case_or_model_files_modified':True}
assert len(impact)==128
(out/'audit_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
table=result[['identifier','raw_population_estimate','raw_population_margin_of_error','lodes_mapped_blocks','included_onset_records_2020_2023','nj_external_outbound_jobs','directly_connected_nj_destinations']]
lines=['| Identifier | ACS population estimate | ACS MOE | LODES blocks | Included onset records | External NJ jobs out | NJ destinations |','|---|---:|---:|---:|---:|---:|---:|']
for row in table.itertuples(index=False,name=None):lines.append('| '+' | '.join(map(str,row))+' |')
report='''# NJ population and geography audit

The eight invalid denominators are all **zero population estimates already present in the original ACS population CSV**. None is a missing population row or a population join-value error. All eight identifiers also appear as ZCTAs in the supplied Gazetteer and LODES block crosswalk.

'''+ '\n'.join(lines)+'''

`B01003_001E` is the population estimate used as the denominator; `B01003_001M` is the margin of error. The value 13 in the MOE field must not be substituted as population. A zero ACS estimate with a nonzero margin of error does not establish that a place has no residents. Positive recorded-onset counts and commuting-job records in the same identifier do not supply a valid population denominator either.

## What this verifies

1. The raw population GEO_ID suffix, NAME identifier and merged ZCTA identifier agree. Merged population equals the original estimate for every audited identifier.
2. The Gazetteer supplies a land-area record for each audited ZCTA; it does not supply population.
3. The LODES crosswalk maps Census blocks to each audited ZCTA in NJ. It does not contain a postal-ZIP correspondence field. Its supplied createdate is 20251202; a 2021 commuting flow year does not mean every geographic input is a 2021 postal correspondence.
4. Consequently, code presence and value transfer are verified, but the residence postal ZIP in the case file is not validated against ZCTA geography by these files. The mapping status must remain unvalidated.

## Consequences and next decisions

The current strict pressure feature is unavailable for 128 destinations linked to these zero-denominator origins, plus eight separate destinations with no external NJ inflow. This explains the 136 excluded pressure-comparison regions. No population values or model files were changed by this audit.

The supplied files cannot repair these eight denominators or establish an authoritative postal ZIP-to-ZCTA correspondence. Do not replace population with MOE, one, workplace jobs, a neighboring area's population or an arbitrary positive value. A compatible alternative population source and a proper postal ZIP–ZCTA correspondence would require additional evidence.

If proceeding using the current sources, a separately labeled sensitivity experiment may restrict origins to valid population denominators, recompute inbound weights over those origins, and retain/report the represented source-weight fraction for every destination. This is a changed pressure definition, not correction of the original complete-source pressure. It must be compared on common regions/dates; destinations with no remaining valid inflow still have undefined pressure.

## Files inspected

- Original ACS: `NJNY_Population 2021.csv.csv`, GEO_ID, NAME, B01003_001E and B01003_001M.
- Gazetteer: `2021_Gaz_zcta_national.txt`, GEOID and ALAND.
- LODES: `nj_xwalk.csv.gz`, tabblk2020, zcta, stusps and locality metadata.
- Existing regional attributes and external route weights under `Weikun`.
- `real_nj_cleaning_outputs/Real NJ COVID Zip.csv`, only ZIP/date and aggregate onset counts.

All calculations are reproducible with the audit notebook. The audit creates separate aggregate outputs and preserves all input files.
'''
(out/'README.md').write_text(report,encoding='utf-8')


In [ ]:
display(result)
display(impact.head(20))
print(out.resolve())